In [1]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os

# 영화 랭킹 정보 조회 및 포스터 이미지 저장하기

In [ ]:
movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

image_dir = 'images'
if not os.path.exists(image_dir):
  os.makedirs(image_dir)

pattern = r'[\\/:"*?<>|]' #파일이나 폴더 이름으로 절대 사용할 수 없는 특수문자 9가지

if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  
  #content > div.wArea.space > div.movieBox > ul > li:nth-child(2) > div > div.movie-title > h3 > a
  movie_title_list = soup.select(".movieBox-list .movie-title a") # 영화 이름 <a> 요소 목록
  movie_image_list = soup.select(".movieBox-list .movieBox-item img") #img 태그 요소
  print(f"수집한 영화 수: {len(movie_title_list)}")

  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    print(movie_title.text, movie_image.get('src'))  #이름 요소의 텍스트, 이미지 요소의 src속성의 값
    
    image_src = movie_image.get('src')
    image_response = requests.get("https://www.moviechart.co.kr" + image_src)
    img = Image.open(BytesIO(image_response.content))
    image_filename = re.sub(pattern, '', movie_title.text) #파일이름 지정, 정규식
    img.save(os.path.join(image_dir, image_filename + ".png"))
    print(movie_title.text, )  
else:
  print("페이지에 연결할 수 없습니다.")

# 영화 포스터 수집 예에서 포스터 원본을 저장

In [ ]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os
from urllib.parse import urlparse, parse_qs

movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

image_dir = 'images2'
if not os.path.exists(image_dir):
  os.makedirs(image_dir)

pattern = r'[\\/:"*?<>|]' 

if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  movie_title_list = soup.select(".movieBox-list .movie-title a")
  movie_image_list = soup.select(".movieBox-list .movieBox-item img")
  print(f"수집한 영화 수: {len(movie_title_list)}")

  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    url = movie_image.get('src')
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    image_src = query_params.get('source', [None])[0]
    # image_response = requests.get('https://www.moviechart.co.kr' + image_src)
    image_response = requests.get(image_src)
    img = Image.open(BytesIO(image_response.content))
    image_filename = re.sub(pattern, '', movie_title.text)
    img.save(os.path.join(image_dir, image_filename + '.png'))
    print(movie_title.text, )
else:
  print("페이지에 연결할 수 없습니다.")